In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
tables = [
    "silver_orders", "silver_payments", "silver_products",
    "silver_customers", "silver_items", "silver_reviews"
]

for t in tables:
    print(f"{t}: {spark.table(t).count():,} rows")

silver_orders: 96,470 rows
silver_payments: 99,437 rows
silver_products: 32,951 rows
silver_customers: 99,441 rows
silver_items: 112,650 rows
silver_reviews: 98,673 rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_revenue_by_category AS
SELECT
    p.product_category,
    COUNT(DISTINCT o.order_id)                          AS total_orders,
    ROUND(SUM(pay.payment_value), 2)                    AS total_revenue,
    ROUND(AVG(pay.payment_value), 2)                    AS avg_order_value,
    ROUND(SUM(pay.payment_value) * 100.0 /
        SUM(SUM(pay.payment_value)) OVER (), 2)         AS revenue_pct
FROM silver_orders      o
JOIN silver_items       i   ON o.order_id   = i.order_id
JOIN silver_products    p   ON i.product_id = p.product_id
JOIN silver_payments    pay ON o.order_id   = pay.order_id
GROUP BY p.product_category
ORDER BY total_revenue DESC

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_delivery_by_state AS
SELECT
    c.customer_state,
    COUNT(o.order_id)                           AS total_orders,
    ROUND(AVG(o.delivery_delay_days), 2)        AS avg_delay_days,
    SUM(o.is_delayed)                           AS delayed_orders,
    ROUND(AVG(o.is_delayed) * 100, 2)           AS delay_pct,
    ROUND(AVG(r.review_score), 2)               AS avg_review_score
FROM silver_orders      o
JOIN silver_customers   c ON o.customer_id  = c.customer_id
JOIN silver_reviews     r ON o.order_id     = r.order_id
GROUP BY c.customer_state
ORDER BY avg_delay_days DESC

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_monthly_revenue AS
WITH monthly AS (
    SELECT
        o.order_month,
        ROUND(SUM(pay.payment_value), 2)    AS revenue,
        COUNT(DISTINCT o.order_id)          AS total_orders
    FROM silver_orders      o
    JOIN silver_payments    pay ON o.order_id = pay.order_id
    GROUP BY o.order_month
)
SELECT
    order_month,
    revenue,
    total_orders,
    LAG(revenue) OVER (ORDER BY order_month)    AS prev_month_revenue,
    ROUND(
        (revenue - LAG(revenue) OVER (ORDER BY order_month)) /
        LAG(revenue) OVER (ORDER BY order_month) * 100
    , 2)                                        AS mom_growth_pct
FROM monthly
ORDER BY order_month

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_customer_segments AS
WITH order_counts AS (
    SELECT
        customer_id,
        COUNT(order_id) AS total_orders
    FROM silver_orders
    GROUP BY customer_id
)
SELECT
    CASE
        WHEN total_orders = 1               THEN 'One-time'
        WHEN total_orders BETWEEN 2 AND 3   THEN 'Returning'
        ELSE                                     'Loyal'
    END                                         AS segment,
    COUNT(*)                                    AS customer_count,
    ROUND(COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (), 2)               AS percentage
FROM order_counts
GROUP BY segment
ORDER BY customer_count DESC

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE gold_review_vs_delay AS
SELECT
    r.review_score,
    r.sentiment,
    COUNT(*)                                    AS order_count,
    ROUND(AVG(o.delivery_delay_days), 2)        AS avg_delay_days,
    ROUND(AVG(o.processing_time_days), 2)       AS avg_processing_days
FROM silver_orders  o
JOIN silver_reviews r ON o.order_id = r.order_id
GROUP BY r.review_score, r.sentiment
ORDER BY r.review_score

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT 'gold_revenue_by_category'  AS table_name, COUNT(*) AS rows FROM gold_revenue_by_category  UNION ALL
SELECT 'gold_delivery_by_state',                  COUNT(*) FROM gold_delivery_by_state             UNION ALL
SELECT 'gold_monthly_revenue',                    COUNT(*) FROM gold_monthly_revenue               UNION ALL
SELECT 'gold_customer_segments',                  COUNT(*) FROM gold_customer_segments             UNION ALL
SELECT 'gold_review_vs_delay',                    COUNT(*) FROM gold_review_vs_delay

table_name,rows
gold_revenue_by_category,72
gold_delivery_by_state,27
gold_monthly_revenue,22
gold_customer_segments,1
gold_review_vs_delay,5


BUSINESS INSIGHTS

In [0]:
%sql
-- Top 5 revenue categories
SELECT product_category, total_orders, total_revenue, revenue_pct
FROM gold_revenue_by_category
LIMIT 5

product_category,total_orders,total_revenue,revenue_pct
bed_bath_table,9272,1674498.41,8.54
health_beauty,8646,1607001.85,8.2
computers_accessories,6529,1541279.4,7.86
furniture_decor,6307,1382526.28,7.05
watches_gifts,5493,1374405.97,7.01


In [0]:
%sql
-- Top 5 most delayed states
SELECT customer_state, total_orders, avg_delay_days, delay_pct, avg_review_score
FROM gold_delivery_by_state
LIMIT 5

customer_state,total_orders,avg_delay_days,delay_pct,avg_review_score
AL,394,-8.82,20.81,3.85
MA,712,-9.65,17.13,3.83
SE,334,-10.07,14.97,3.91
ES,1969,-10.68,10.41,4.08
CE,1273,-10.85,13.67,3.94


In [0]:
%sql
-- Customer segments
SELECT * FROM gold_customer_segments

segment,customer_count,percentage
One-time,96470,100.00


In [0]:
%sql
-- Review score vs delay — key insight
SELECT * FROM gold_review_vs_delay

review_score,sentiment,order_count,avg_delay_days,avg_processing_days
1,Negative,9351,-4.03,3.77
2,Negative,2920,-8.66,3.07
3,Neutral,7916,-10.76,2.63
4,Positive,18894,-12.38,2.29
5,Positive,56743,-13.38,1.96


In [0]:
import os

export_path = "/Workspace/Users/prarthanabg7@gmail.com/E-commerce/03_Exports/"

gold_tables = [
    "gold_revenue_by_category",
    "gold_delivery_by_state",
    "gold_monthly_revenue",
    "gold_customer_segments",
    "gold_review_vs_delay"
]

for table in gold_tables:
    df = spark.table(table).toPandas()
    df.to_csv(f"{export_path}{table}.csv", index=False)
    print(f"✅ {table}: {len(df):,} rows exported")

✅ gold_revenue_by_category: 72 rows exported
✅ gold_delivery_by_state: 27 rows exported
✅ gold_monthly_revenue: 22 rows exported
✅ gold_customer_segments: 1 rows exported
✅ gold_review_vs_delay: 5 rows exported
